# SVM Model Training with Performance Metrics and ROC Curve
This notebook trains a Support Vector Machine (SVM) model on synthetic foot ulcer risk dataset and evaluates its performance with comprehensive metrics, confusion matrix, and ROC curve.

## 1. Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, confusion_matrix, classification_report,
                             roc_curve, auc, roc_auc_score)
import pickle
import os
import warnings

warnings.filterwarnings('ignore')

# Set style for better visualizations
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

## 2. Load Synthetic Dataset

In [ ]:
# Load the synthetic dataset
dataset_path = '../Synthetic_Data/synthetic_foot_ulcer_dataset_RISK.csv'
df = pd.read_csv(dataset_path)

print("Dataset Shape:", df.shape)
print("\nFirst few rows:")
print(df.head())
print("\nDataset Info:")
print(df.info())
print("\nDataset Statistics:")
print(df.describe())

## 3. Explore and Prepare Data

In [ ]:
# Check for missing values
print("Missing Values:")
print(df.isnull().sum())

# Check class distribution
print("\n\nClass Distribution:")
print(df['label'].value_counts())
print("\nClass Distribution (%):")
print(df['label'].value_counts(normalize=True) * 100)

# Prepare features and target
# Drop the 'label' column to get features
X = df.drop(['label'], axis=1)
y = df['label']

print(f"\nFeatures shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nFeature columns: {list(X.columns)}")

## 4. Split Data into Train and Test Sets

In [ ]:
# Split data into training (80%) and testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

print(f"Training set size: {X_train.shape[0]} samples")
print(f"Testing set size: {X_test.shape[0]} samples")
print(f"\nTraining set class distribution:")
print(y_train.value_counts())
print(f"\nTesting set class distribution:")
print(y_test.value_counts())

# Scale features for better SVM performance
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\nFeatures scaled successfully!")
print(f"Training data shape: {X_train_scaled.shape}")
print(f"Testing data shape: {X_test_scaled.shape}")

## 5. Train SVM Model

In [ ]:
# Initialize and train SVM model with RBF kernel
# RBF kernel is effective for binary classification with non-linear boundaries
svm_model = SVC(
    kernel='rbf',  # Radial Basis Function kernel
    C=1.0,  # Regularization parameter
    gamma='scale',  # Kernel coefficient
    probability=True,  # Enable probability estimates for ROC curve
    random_state=42
)

print("Training SVM Model...")
print("Model Parameters:")
print(f"  - Kernel: {svm_model.kernel}")
print(f"  - C (Regularization): {svm_model.C}")
print(f"  - Gamma: {svm_model.gamma}")

# Train the model
svm_model.fit(X_train_scaled, y_train)

print(f"\nModel Training Complete!")
print(f"Number of support vectors: {len(svm_model.support_vectors_)}")
print(f"Support vectors ratio: {len(svm_model.support_vectors_) / len(X_train_scaled) * 100:.2f}%")

## 6. Generate Predictions

In [ ]:
# Generate predictions on test set
y_pred = svm_model.predict(X_test_scaled)

# Get probability predictions for ROC curve
y_pred_proba = svm_model.predict_proba(X_test_scaled)[:, 1]

# Also get predictions on training set for comparison
y_train_pred = svm_model.predict(X_train_scaled)

print("Predictions Generated!")
print(f"\nTest Set Predictions sample (first 20):")
print(y_pred[:20])
print(f"\nUnique predictions: {np.unique(y_pred)}")

## 7. Calculate Performance Metrics

In [ ]:
# Calculate performance metrics for test set
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_proba)

# Also calculate for training set
train_accuracy = accuracy_score(y_train, y_train_pred)
train_precision = precision_score(y_train, y_train_pred)
train_recall = recall_score(y_train, y_train_pred)
train_f1 = f1_score(y_train, y_train_pred)
train_roc_auc = roc_auc_score(y_train, svm_model.predict_proba(X_train_scaled)[:, 1])

# Print all performance metrics
print("="*70)
print("SUPPORT VECTOR MACHINE (SVM) MODEL - PERFORMANCE METRICS")
print("="*70)

print("\n📊 TEST SET PERFORMANCE:")
print("-"*70)
print(f"✓ Accuracy:  {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"✓ Precision: {precision:.4f} ({precision*100:.2f}%)")
print(f"✓ Recall:    {recall:.4f} ({recall*100:.2f}%)")
print(f"✓ F1-Score:  {f1:.4f}")
print(f"✓ ROC-AUC:   {roc_auc:.4f}")

print("\n📊 TRAINING SET PERFORMANCE (for comparison):")
print("-"*70)
print(f"✓ Accuracy:  {train_accuracy:.4f} ({train_accuracy*100:.2f}%)")
print(f"✓ Precision: {train_precision:.4f} ({train_precision*100:.2f}%)")
print(f"✓ Recall:    {train_recall:.4f} ({train_recall*100:.2f}%)")
print(f"✓ F1-Score:  {train_f1:.4f}")
print(f"✓ ROC-AUC:   {train_roc_auc:.4f}")

print("\n" + "="*70)
print("DETAILED CLASSIFICATION REPORT (TEST SET)")
print("="*70)
print(classification_report(y_test, y_pred, target_names=['No Risk (0)', 'Risk (1)']))
print("="*70)

## 8. Display Confusion Matrix

In [ ]:
# Create confusion matrix
cm = confusion_matrix(y_test, y_pred)

# Extract values
tn, fp, fn, tp = cm.ravel()

# Plot confusion matrix
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Heatmap
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['No Risk (0)', 'Risk (1)'],
            yticklabels=['No Risk (0)', 'Risk (1)'],
            cbar_kws={'label': 'Count'},
            ax=ax1,
            annot_kws={'size': 16, 'weight': 'bold'})
ax1.set_ylabel('True Label', fontsize=12, fontweight='bold')
ax1.set_xlabel('Predicted Label', fontsize=12, fontweight='bold')
ax1.set_title('Confusion Matrix - SVM Model', fontsize=14, fontweight='bold')

# Detailed metrics from confusion matrix
specificity = tn / (tn + fp) if (tn + fp) != 0 else 0
sensitivity = tp / (tp + fn) if (tp + fn) != 0 else 0
false_positive_rate = fp / (fp + tn) if (fp + tn) != 0 else 0
false_negative_rate = fn / (fn + tp) if (fn + tp) != 0 else 0

# Create a text summary
metrics_text = f"""
CONFUSION MATRIX BREAKDOWN:

True Negatives (TN):  {tn:4d}  [Correctly predicted No Risk]
True Positives (TP):  {tp:4d}  [Correctly predicted Risk]
False Positives (FP): {fp:4d}  [Incorrectly predicted Risk]
False Negatives (FN): {fn:4d}  [Incorrectly predicted No Risk]

DERIVED METRICS:
Sensitivity (Recall):     {sensitivity:.4f}
Specificity:              {specificity:.4f}
False Positive Rate:      {false_positive_rate:.4f}
False Negative Rate:      {false_negative_rate:.4f}
"""

ax2.text(0.1, 0.5, metrics_text, fontsize=11, family='monospace',
         verticalalignment='center', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
ax2.axis('off')

plt.tight_layout()
plt.show()

print("\n" + "="*70)
print("CONFUSION MATRIX ANALYSIS")
print("="*70)
print(metrics_text)

## 9. Plot ROC Curve

In [ ]:
# Calculate ROC curve
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
roc_auc_value = auc(fpr, tpr)

# Also calculate for training set
fpr_train, tpr_train, _ = roc_curve(y_train, svm_model.predict_proba(X_train_scaled)[:, 1])
roc_auc_train = auc(fpr_train, tpr_train)

# Plot ROC Curve
plt.figure(figsize=(10, 8))

# Plot test ROC curve
plt.plot(fpr, tpr, color='darkorange', lw=2.5, 
         label=f'Test ROC Curve (AUC = {roc_auc_value:.4f})', marker='o', markersize=4)

# Plot training ROC curve for comparison
plt.plot(fpr_train, tpr_train, color='green', lw=2.5, linestyle='--',
         label=f'Training ROC Curve (AUC = {roc_auc_train:.4f})', marker='s', markersize=4)

# Plot diagonal reference line (random classifier)
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Classifier (AUC = 0.50)')

# Formatting
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (1 - Specificity)', fontsize=12, fontweight='bold')
plt.ylabel('True Positive Rate (Sensitivity)', fontsize=12, fontweight='bold')
plt.title('ROC Curve - SVM Model\n(Receiver Operating Characteristic Curve)', 
          fontsize=14, fontweight='bold')
plt.legend(loc="lower right", fontsize=11)
plt.grid(True, alpha=0.3)

# Add annotations for key points
plt.scatter([0], [1], marker='*', s=500, color='red', label='Ideal Classifier', zorder=5)

plt.tight_layout()
plt.show()

print("\n" + "="*70)
print("ROC CURVE ANALYSIS")
print("="*70)
print(f"Test Set ROC-AUC Score:     {roc_auc_value:.4f}")
print(f"Training Set ROC-AUC Score: {roc_auc_train:.4f}")
print(f"\nInterpretation:")
print(f"  - AUC = 1.0: Perfect classifier")
print(f"  - AUC = 0.5: Random classifier (diagonal line)")
print(f"  - AUC > 0.5: Better than random")
print(f"  - Current AUC: {roc_auc_value:.4f} - {'Excellent' if roc_auc_value > 0.9 else 'Good' if roc_auc_value > 0.8 else 'Fair'}")
print("="*70)

## 10. Save Trained Model

In [ ]:
# Create models directory if it doesn't exist
models_dir = './models'
if not os.path.exists(models_dir):
    os.makedirs(models_dir)
    print(f"Created models directory: {models_dir}")

# Save the SVM model
model_path = os.path.join(models_dir, 'svm_model.pkl')
with open(model_path, 'wb') as f:
    pickle.dump(svm_model, f)
print(f"✓ SVM Model saved to: {model_path}")

# Save the scaler
scaler_path = os.path.join(models_dir, 'scaler.pkl')
with open(scaler_path, 'wb') as f:
    pickle.dump(scaler, f)
print(f"✓ Feature Scaler saved to: {scaler_path}")

# Save model metadata
metadata = {
    'model_type': 'SVM (Support Vector Machine)',
    'kernel': 'RBF',
    'test_accuracy': float(accuracy),
    'test_precision': float(precision),
    'test_recall': float(recall),
    'test_f1_score': float(f1),
    'test_roc_auc': float(roc_auc),
    'train_accuracy': float(train_accuracy),
    'train_roc_auc': float(train_roc_auc),
    'feature_names': list(X.columns),
    'n_features': len(X.columns),
    'n_training_samples': len(X_train),
    'n_testing_samples': len(X_test),
}

import json
metadata_path = os.path.join(models_dir, 'svm_metadata.json')
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=4)
print(f"✓ Model metadata saved to: {metadata_path}")

print("\n" + "="*70)
print("MODEL SAVE SUMMARY")
print("="*70)
print(f"All files saved in: {os.path.abspath(models_dir)}")
print(f"\nSaved Files:")
print(f"  1. svm_model.pkl          - Trained SVM model")
print(f"  2. scaler.pkl             - Feature scaler for preprocessing")
print(f"  3. svm_metadata.json      - Model metadata and performance info")
print("="*70)

print("\n✅ Model Training and Saving Complete!")
print(f"\nFinal Test Accuracy: {accuracy*100:.2f}%")
print(f"Final ROC-AUC Score: {roc_auc:.4f}")